In [1]:
%run ./config_api_acto

StatementMeta(, 2caae215-2bdb-4b3f-8241-9e8b2012a09f, 3, Finished, Available, Finished)

In [2]:
import requests
import json
import pandas as pd
from datetime import datetime, timezone
import numpy as np
import json
from pathlib import Path
from unidecode import unidecode

TOKEN = TOKEN_MAUA

HOJE = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.000Z")

HEADERS = {
        "Accept": "application/json, text/plain, */*",
        "Authorization": f"Bearer {TOKEN}",
        "User-Agent": "Mozilla/5.0 ...",
        "Content-Type": "application/json",
}


def import_json_payload(arquivo: str):
    path_json = "/lakehouse/default/Files/payload/" + arquivo

    with open(path_json, "r", encoding="utf-8") as f:
        json_dict = json.load(f)

    json_config = json.dumps(json_dict, ensure_ascii=False)
    return json_config


def obter_dados_etapa_atual(TOKEN: str, codigo_catalogo: list) -> pd.DataFrame:

    url = "https://actogestaoapi-gdhrfgdfc8bbe8hs.brazilsouth-01.azurewebsites.net/api/RelatoriosEtapa/ObterTempoEtapaRelatorio"
    payload_etapa = {
        "codCatalogos": codigo_catalogo,
        "dataInicio": "2024-01-01T00:00:00.000Z",
        "dataFim": HOJE,
        "ativo": 1,
    }
    r = requests.post(url, json=payload_etapa, headers=HEADERS, timeout=60)
    r.raise_for_status()
    data = r.json()
    df = pd.DataFrame(data if isinstance(data, list) else data.get("data", []))
    return df


def fetch_tabela(payload_str: str) -> pd.DataFrame:
    """Recebe o JSON do --data-raw como string e devolve um DataFrame com os dados."""
    
    url_dados = "https://actogestaoapi-gdhrfgdfc8bbe8hs.brazilsouth-01.azurewebsites.net/api/Tabela/VisualizarDadosIntermediarios"
    
    config = json.loads(payload_str)
    resp = requests.post(url_dados, headers=HEADERS, json=config)
    print("Status:", resp.status_code)
    
    if resp.status_code != 200:
        print("Corpo da resposta:", resp.text[:500])
        raise SystemExit("❌ Erro ao chamar API")
    
    data = resp.json()
    
    lista_final = []
    if "data" in data and isinstance(data["data"], list):
        for item in data["data"]:
            dados_dict = item.get("dados", {})
            if isinstance(dados_dict, dict):
                for key, lista in dados_dict.items():
                    if isinstance(lista, list):
                        lista_final.extend(lista)
    if not lista_final:
        print("✅ Requisição OK, mas sem linhas.")
        return pd.DataFrame()
    
    df = pd.DataFrame(lista_final)
    print(f"✅ Linhas em pandas: {len(df)}")
    return df


def adicionar_etapa_atual_2(
    df_etapas: pd.DataFrame, df_solicitacoes: pd.DataFrame
) -> pd.DataFrame:
    temp = df_etapas.groupby("seqFluxo", as_index=False)["dataAtenderEtapa"].max()
    temp_etapa_os = (
        df_etapas.merge(temp, on=["seqFluxo", "dataAtenderEtapa"], how="inner")
        .groupby("seqFluxo", as_index=False)
        .size()
    )
    lista = (
        temp_etapa_os.sort_values("size", ascending=False)
        .query("size > 1")["seqFluxo"]
        .tolist()
    )
    temp = temp.loc[~temp["seqFluxo"].isin(lista)].copy()
    df_etapa_atual = df_etapas.merge(
        temp, on=["seqFluxo", "dataAtenderEtapa"], how="inner"
    )
    df_etapa_comeca_mesmo_tempo = df_etapas.query(
        "seqFluxo in @lista and dataEtapaFim.isna()"
    ).sort_values(["seqFluxo", "dataAtenderEtapa"])
    df_etapa_atual2 = pd.concat([df_etapa_atual, df_etapa_comeca_mesmo_tempo], axis=0)
    df_etapa_atual2 = df_etapa_atual2[["seqFluxo", "etapa", "executor"]].copy()
    df_solicitacoes = df_solicitacoes.rename(columns={"Nº Solicitação": "seqFluxo"})
    df_solicitacoes = df_solicitacoes.astype({"seqFluxo": "int64"})
    df_solicitacoes["Data Criação"] = pd.to_datetime(
        df_solicitacoes["Data Criação"], format="ISO8601"
    )
    df_solicitacoes = df_solicitacoes.merge(df_etapa_atual2, on="seqFluxo", how="left")

    return df_solicitacoes


def extrair_tabela_acto_gestao(arquivo: str, lista_cod_catalogo: list):

    df_etapas = obter_dados_etapa_atual(TOKEN, lista_cod_catalogo)

    df_solicitacoes = fetch_tabela(import_json_payload(arquivo))

    strings_procuradas = ["logradouro", "Logradouro", "Número", "Agendamento"]

    colunas_encontradas = [
        col
        for col in df_solicitacoes.columns
        if any(s in col for s in strings_procuradas)
    ]

    df_solicitacoes = df_solicitacoes.drop(columns=colunas_encontradas)

    return df_solicitacoes, df_etapas


def aplicar_bfill(df_solicitacoes_segov, coluna: str):
    n_solicitacao = df_solicitacoes_segov.filter(like=coluna).columns
    
    if len(n_solicitacao) == 0:
        print(f"⚠️ Coluna '{coluna}' não encontrada")
        return df_solicitacoes_segov
    
    df_solicitacoes_segov[coluna] = (
        df_solicitacoes_segov[n_solicitacao].bfill(axis=1).iloc[:, 0]
    )
    df_solicitacoes_segov = df_solicitacoes_segov.drop(columns=n_solicitacao)
    
    return df_solicitacoes_segov


def tratar_nome_coluna(df_solicitacoes_etapa):

    rename_map_bi_asis = {
        "seqfluxo": "os",
        "etapa": "etapa_atual",
        "data_criacao": "data_solicitacao"
    }
    df_solicitacoes_etapa = df_solicitacoes_etapa.rename(columns=rename_map_bi_asis)

    return df_solicitacoes_etapa

StatementMeta(, 2caae215-2bdb-4b3f-8241-9e8b2012a09f, 4, Finished, Available, Finished)

In [3]:



def main():

    lista_1 = [
        7013,
        7094,
        7453,
        7545,
        7653,
        7674,
        7676,
        8064,
        8494,
        9484,
        10074,
        11574,
        11804,
        11814,
        12726,
    ]

    df_solicitacoes, df_etapas = extrair_tabela_acto_gestao(
        arquivo="payload_maua_meio_ambiente.json", lista_cod_catalogo=lista_1
    )

    return df_solicitacoes, df_etapas


# execução
df_solicitacoes, df_etapas = main()

coluna_os_rename = df_solicitacoes.filter(like="N° da Solicitação").columns[0]
df_solicitacoes = df_solicitacoes.drop(columns=[coluna_os_rename])

df_solicitacoes.columns = df_solicitacoes.columns.str.replace(":", "")

COLUNAS = [
    "Nº Solicitação",
    "Serviço",
    "Data Finalização",
    "Status",
    "Bairro",
    "Solicitante",
    "Data Criação",
    "CPF",
    "CNPJ",
    "Nome",
    "Inscrição Fiscal do imóvel",
]
for c in COLUNAS:
    df_solicitacoes = aplicar_bfill(df_solicitacoes, c)


df_solicitacoes_final = df_solicitacoes[COLUNAS].copy()
df_solicitacoes_final = adicionar_etapa_atual_2(df_etapas, df_solicitacoes_final)

df_solicitacoes_final.columns = df_solicitacoes_final.columns.str.replace("º", "").str.lower().str.replace(" ", "_")
df_solicitacoes_final.columns = [unidecode(col) for col in df_solicitacoes_final.columns]

df_solicitacoes_final = tratar_nome_coluna(df_solicitacoes_final)

for col in ["data_solicitacao", "data_finalizacao"]:
    df_solicitacoes_final[col] = pd.to_datetime(
        df_solicitacoes_final[col], format="ISO8601"
    )

# etapas
df_etapas = df_etapas.drop(columns=["codEtapa", "servico", "tempoExecucao", "tempoExecucaoHoras", "notifications", "isValid"])

cols_data = df_etapas.filter(like="data").columns
for col in cols_data:
    df_etapas[col] = pd.to_datetime(df_etapas[col], format="ISO8601")


df_etapas = df_etapas.rename(columns={
    "seqFluxo": "os",
    "dataCriacaoOS": "data_solicitacao",
    "dataFinalizacaoOS": "data_finalizacao",
    "dataEtapaInicio": "data_etapa_inicio",
    "dataEtapaFim": "data_etapa_fim",
    "dataAtenderEtapa": "data_atender_etapa"
})

StatementMeta(, 2caae215-2bdb-4b3f-8241-9e8b2012a09f, 5, Finished, Available, Finished)

Status: 200
✅ Linhas em pandas: 3229


In [6]:
sdf_etapas = spark.createDataFrame(df_etapas)
sdf_solicitacoes_final = spark.createDataFrame(df_solicitacoes_final)

(
    sdf_etapas
    .write.mode("overwrite")
    .format("delta")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_maua_meio_ambiente_etapas")
)
(
    sdf_solicitacoes_final
    .write.mode("overwrite")
    .format("delta")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_maua_meio_ambiente_solicitacoes")
)

StatementMeta(, 2caae215-2bdb-4b3f-8241-9e8b2012a09f, 8, Finished, Available, Finished)